# CLIP linear probe baseline

The decision model has to beat this. Following UniversalFakeDetect (Ojha et al., CVPR 2023), it trains a logistic regression on frozen CLIP ViT-L/14 image features and scores the same held-out subjects, under the same protocol (`face_decision_lab.protocol`), as `train_decision_model.ipynb`.

**Runtime:** Colab Pro with an L4 or A100 GPU. **Needs** the Colab secret `HF_TOKEN` to read the private dataset.

### Configuration

In [ ]:
REPO_SOURCE = "git+https://github.com/paulohenriquevn/face-decision-lab@workspace"
HF_DATASET = "paulohenriquevn/face-decision-data"
CLIP_MODEL = "openai/clip-vit-large-patch14"
BATCH_SIZE = 256
# Probe training sizes for a learning curve. None is the whole train side; 1920 is the same rows
# (same seed, same capped()) the 60-step decision model trained on.
PROBE_TRAIN_SIZES = (1920, None)
# Protocols to score; see face_decision_lab.protocol.PROTOCOLS.
PROTOCOL_NAMES = ("subject_holdout", "unseen_text2img", "unseen_insight", "unseen_inpainting")

### Installation

In [ ]:
%%capture
!pip install -q "face-decision-lab @ {REPO_SOURCE}"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token = userdata.get("HF_TOKEN"))

### Data, split by the shared protocol

In [ ]:
from collections import Counter
from face_decision_lab.hub import load_hub_samples
from face_decision_lab.protocol import CALIBRATION_FRACTION, SEED, TEST_FRACTION
from face_decision_lab.splits import standard_split

samples, hub_images = load_hub_samples(HF_DATASET)
images = {s.image_path: image for s, image in zip(samples, hub_images, strict = True)}
# Features are extracted once for the whole train and test sides; each protocol selects from them.
all_train, _, all_test = standard_split(samples, TEST_FRACTION, CALIBRATION_FRACTION, SEED)
for name, part in [("train", all_train), ("test", all_test)]:
    print(name, len(part), dict(Counter(s.attack_type for s in part)))

### Frozen CLIP features

In [ ]:
import numpy as np
import torch
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor
from face_decision_lab.rows import open_image

clip = CLIPModel.from_pretrained(CLIP_MODEL, torch_dtype = torch.float16).to("cuda").eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL)

@torch.no_grad()
def clip_features(part):
    out = []
    with ThreadPoolExecutor(max_workers = 8) as pool:
        for start in tqdm(range(0, len(part), BATCH_SIZE)):
            batch = list(pool.map(lambda s: open_image(images[s.image_path]), part[start:start + BATCH_SIZE]))
            pixels = clip_processor(images = batch, return_tensors = "pt")["pixel_values"].to("cuda", torch.float16)
            # The explicit path works across transformers versions; get_image_features changed its return type in 5.x.
            pooled = clip.vision_model(pixel_values = pixels).pooler_output
            features = clip.visual_projection(pooled)
            out.append(torch.nn.functional.normalize(features, dim = -1).float().cpu().numpy())
    return np.concatenate(out)

every = all_train + all_test
matrix = clip_features(every)
features = {s.image_path: row for s, row in zip(every, matrix, strict = True)}
print(matrix.shape)

### Probe and report

In [ ]:
import json
from face_decision_lab.baselines import LinearProbe
from face_decision_lab.protocol import evaluation_report, protocol_split
from face_decision_lab.splits import capped

def stack(part):
    return np.stack([features[s.image_path] for s in part])

reports = {}
for protocol in PROTOCOL_NAMES:
    train_samples, _, test_samples = protocol_split(samples, protocol)
    for size in PROBE_TRAIN_SIZES:
        subset = capped(train_samples, size, SEED)
        probe = LinearProbe(seed = SEED).fit(stack(subset), np.array([not s.is_bona_fide for s in subset]))
        reports[f"{protocol}/{len(subset)}"] = evaluation_report(test_samples, probe.attack_scores(stack(test_samples)).tolist())
print(json.dumps(reports, indent = 2))